# Data Understanding

In [ ]:
import os
import fitz  # PyMuPDF
import pdfplumber
import easyocr
import requests
from bs4 import BeautifulSoup
import json
import numpy as np

# =========================
# CONFIG
# =========================
RAW_PDF_DIR = 'data/raw_pdfs' # ganti jadi 'data/raw_pdfs'
FAQ_PDF_PATH = 'data/custom_data/faq_lengkap.pdf'
URLS_PATH = 'data/custom_data/urls.txt'
OUTPUT_FILE = 'data/knowledge_base.json'

corpus_data = []

# EasyOCR init
reader = easyocr.Reader(['id', 'en'])


# =========================
# 1. PDF NATIF (TEKS)
# =========================
def extract_native_pdf(pdf_path):
    print(f"[FAQ] {pdf_path}")
    text_content = ""

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                text_content += text + "\n"

    corpus_data.append({
        "source": os.path.basename(pdf_path),
        "type": "native_pdf",
        "content": text_content.strip()
    })


# =========================
# 2. OCR EASYOCR (FIXED)
# =========================
def ocr_with_easyocr(pdf_path):
    doc = fitz.open(pdf_path)
    text_content = ""

    for page in doc:
        pix = page.get_pixmap(dpi=300)

        # 🔥 FIX: convert langsung ke numpy array (bukan PIL)
        img = np.frombuffer(pix.samples, dtype=np.uint8)
        img = img.reshape(pix.height, pix.width, pix.n)

        # kalau grayscale → convert ke 3 channel
        if pix.n == 1:
            img = np.stack([img] * 3, axis=-1)

        result = reader.readtext(img, detail=0)
        text_content += " ".join(result) + "\n"

    return text_content


# =========================
# 3. PROCESS PDF
# =========================
def process_mixed_pdfs(directory):
    for filename in os.listdir(directory):
        if not filename.endswith(".pdf"):
            continue

        pdf_path = os.path.join(directory, filename)
        print(f"\n[CHECK] {filename}")

        is_scanned = True
        text_content = ""

        # detect native / scanned
        try:
            with pdfplumber.open(pdf_path) as pdf:
                if len(pdf.pages) > 0:
                    first_page = pdf.pages[0].extract_text()

                    if first_page and len(first_page.strip()) > 50:
                        is_scanned = False
        except Exception as e:
            print(f"Detect error {filename}: {e}")

        # process
        if not is_scanned:
            print(f"[NATIVE] {filename}")

            with pdfplumber.open(pdf_path) as pdf:
                for page in pdf.pages:
                    text = page.extract_text()
                    if text:
                        text_content += text + "\n"

        else:
            print(f"[OCR] {filename}")
            try:
                text_content = ocr_with_easyocr(pdf_path)
            except Exception as e:
                print(f"Gagal OCR {filename}: {e}")

        corpus_data.append({
            "source": filename,
            "type": "pdf",
            "content": text_content.strip()
        })


# =========================
# 4. WEB SCRAPING
# =========================
def scrape_urls(url_file):
    with open(url_file, 'r', encoding='utf-8') as f:
        urls = f.readlines()

    for url in urls:
        url = url.strip()
        if not url:
            continue

        print(f"[WEB] {url}")

        try:
            response = requests.get(url, timeout=10)
            soup = BeautifulSoup(response.content, 'html.parser')

            for tag in soup(["script", "style", "nav", "footer"]):
                tag.extract()

            text = soup.get_text(separator=' ', strip=True)

            corpus_data.append({
                "source": url,
                "type": "website",
                "content": text
            })

        except Exception as e:
            print(f"Gagal {url}: {e}")


# =========================
# 5. MAIN PIPELINE
# =========================
if __name__ == "__main__":
    extract_native_pdf(FAQ_PDF_PATH)
    process_mixed_pdfs(RAW_PDF_DIR)
    scrape_urls(URLS_PATH)

    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(corpus_data, f, ensure_ascii=False, indent=4)

    print("\n✅ Knowledge base selesai dibuat!")

[FAQ] /content/drive/MyDrive/Nlp-data/data/custom_data/faq_lengkap.pdf

[CHECK] Kalender_ Akdemik 2025_2026.pdf
[OCR] Kalender_ Akdemik 2025_2026.pdf

[CHECK] Peraturan-Rektor-Kemahasiswaan.pdf
[OCR] Peraturan-Rektor-Kemahasiswaan.pdf

[CHECK] RENSTRA-FMIPA-2020-2024.pdf
[NATIVE] RENSTRA-FMIPA-2020-2024.pdf

[CHECK] SSertifikat-87748-9dd0be2a054e419e82e6538ffda603f9_sign.pdf
[NATIVE] SSertifikat-87748-9dd0be2a054e419e82e6538ffda603f9_sign.pdf

[CHECK] SSK-87748-9dd0be2a054e419e82e6538ffda603f9_sign.pdf
[NATIVE] SSK-87748-9dd0be2a054e419e82e6538ffda603f9_sign.pdf

[CHECK] Surat_SK_Akreditasi-ILMU-KOMPUTER-UNIVERSITAS-HALU-OLEO-2024.pdf
[NATIVE] Surat_SK_Akreditasi-ILMU-KOMPUTER-UNIVERSITAS-HALU-OLEO-2024.pdf

[CHECK] SK-STRUKTUR-ORGANISASI-SERTA-DESKRIPSI-TUGAS-TANGGUNG-JAWAB.pdf
[OCR] SK-STRUKTUR-ORGANISASI-SERTA-DESKRIPSI-TUGAS-TANGGUNG-JAWAB.pdf

[CHECK] 2025_16_September-Surat-Surat-Jadwal-Informasi-Audit-Mutu-Internal-AMI-Siklus-Ke-7.pdf
[OCR] 2025_16_September-Surat-Surat-Jadwal-I

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Data Preprocessing

In [ ]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json

# =========================
# CONFIG PATH
# =========================
# File sumber (hasil Data Gathering yang sudah selesai)
INPUT_FILE = 'data/knowledge_base.json'

# File baru untuk menyimpan hasil potongan teks
OUTPUT_CHUNKS_FILE = 'data/processed_chunks.json'

# 1. Fungsi Pembersihan
def clean_text(text):
    if not text:
        return ""

    # Hapus spasi dan baris baru (newline) yang berlebih
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r'\s+', ' ', text)

    # Hapus karakter aneh/simbol non-ASCII yang sering muncul dari OCR PDF
    text = re.sub(r'[^\w\s.,?!;:()\[\]\-"\']', '', text)

    return text.strip()

# 2. Fungsi Chunking
def process_and_chunk_corpus(corpus_data):
    # Inisialisasi pemotong teks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,   # Maksimal karakter per potongan
        chunk_overlap=200, # Irisan karakter antar potongan agar konteks tidak hilang
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""] # Prioritas pemotongan dari paragraf -> kalimat -> kata
    )

    processed_chunks = []

    for item in corpus_data:
        # Bersihkan teks dulu
        cleaned_text = clean_text(item["content"])

        # Potong-potong teksnya
        chunks = text_splitter.split_text(cleaned_text)

        # Simpan kembali dengan metadatanya
        for idx, chunk in enumerate(chunks):
            processed_chunks.append({
                "source": item["source"],
                "type": item["type"],
                "chunk_id": f"{item['source']}_chunk_{idx}",
                "text": chunk
            })

    return processed_chunks

# =========================
# 3. MAIN PIPELINE
# =========================
if __name__ == "__main__":
    print(f"Membaca data dari: {INPUT_FILE}")

    # BACA DATA (Gunakan 'r' untuk read)
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        corpus_data = json.load(f)

    print(f"Berhasil memuat {len(corpus_data)} dokumen. Memulai proses pembersihan dan chunking...")

    # Proses datasetnya
    final_chunks = process_and_chunk_corpus(corpus_data)

    # SIMPAN HASIL CHUNKING (Gunakan file baru agar data lama tidak tertimpa)
    with open(OUTPUT_CHUNKS_FILE, 'w', encoding='utf-8') as f:
        json.dump(final_chunks, f, ensure_ascii=False, indent=4)

    print(f"✅ Selesai! Berhasil memecah teks menjadi {len(final_chunks)} chunks.")
    print(f"Hasil preprocessing disimpan di: {OUTPUT_CHUNKS_FILE}")

Membaca data dari: /content/drive/MyDrive/Nlp-data/data/knowledge_base.json
Berhasil memuat 98 dokumen. Memulai proses pembersihan dan chunking...
✅ Selesai! Berhasil memecah teks menjadi 1219 chunks.
Hasil preprocessing disimpan di: /content/drive/MyDrive/Nlp-data/data/processed_chunks.json


# Representasi Teks


In [28]:
import json
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# ==========================================
# CONFIG PATH
# =========================
# Path JSON hasil chunking dari langkah sebelumnya
CHUNKS_FILE = "/content/drive/MyDrive/Nlp-data/data/processed_chunks.json"
# Path untuk menyimpan Vector Database
VECTOR_DB_DIR = "/content/drive/MyDrive/Nlp-data/data/chroma_db"

print("1. Membaca data chunks...")
with open(CHUNKS_FILE, 'r', encoding='utf-8') as f:
    chunks_data = json.load(f)

print(f"Berhasil memuat {len(chunks_data)} potongan teks.")

# 2. Siapkan Model Embedding
print("\n2. Mengunduh/Memuat model embedding...")
embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# 3. Ekstrak teks dan metadata
texts = [item["text"] for item in chunks_data]
metadatas = [{"source": item["source"], "type": item["type"]} for item in chunks_data]

# 4. Injeksi ke ChromaDB
print("\n3. Membangun Vector Database (Proses ini memakan waktu beberapa menit)...")
vector_db = Chroma.from_texts(
    texts=texts,
    embedding=embeddings_model,
    metadatas=metadatas,
    persist_directory=VECTOR_DB_DIR
)

print(f"\n✅ DATABASE SELESAI DIBUAT! Data berhasil disimpan di {VECTOR_DB_DIR}.")

1. Membaca data chunks...
Berhasil memuat 1219 potongan teks.

2. Mengunduh/Memuat model embedding...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



3. Membangun Vector Database (Proses ini memakan waktu beberapa menit)...

✅ DATABASE SELESAI DIBUAT! Data berhasil disimpan di /content/drive/MyDrive/Nlp-data/data/chroma_db.


# Pemodelan

In [29]:
import os
from getpass import getpass
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# ==========================================
# SETUP GOOGLE GEMINI API KEY
# ==========================================
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass("Masukkan Google Gemini API Key Anda: ")

# ==========================================
# 1. LOAD VECTOR DATABASE
# ==========================================
VECTOR_DB_DIR = "/content/drive/MyDrive/Nlp-data/data/chroma_db"
embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

print("Memuat Vector Database...")
vector_db = Chroma(persist_directory=VECTOR_DB_DIR, embedding_function=embeddings_model)

# Gunakan Vector DB sebagai retriever
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

# ==========================================
# 2. INISIALISASI LLM (GEMINI)
# ==========================================
print("Inisialisasi Gemini API...")
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# ==========================================
# 3. PROMPT TEMPLATE
# ==========================================
prompt_template = """
Kamu adalah asisten AI resmi untuk Fakultas Matematika dan Ilmu Pengetahuan Alam (FMIPA) Universitas Halu Oleo.
Gunakan HANYA potongan konteks berikut untuk menjawab pertanyaan pengguna.
Jika jawabannya tidak ada di dalam konteks, katakan dengan sopan bahwa kamu belum memiliki informasi tersebut. Jangan mengarang informasi (halusinasi).
Gunakan bahasa Indonesia yang ramah, jelas, dan informatif.

Konteks:
{context}

Pertanyaan: {question}

Jawaban:
"""
prompt = ChatPromptTemplate.from_template(prompt_template)

# ==========================================
# 4. MEMBANGUN CHAIN DENGAN LCEL (MODERN WAY)
# ==========================================
# Fungsi helper untuk menggabungkan teks dari dokumen yang ditarik oleh Retriever
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Sintaks pipe (|) khas LCEL. Jauh lebih bersih!
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser() # Langsung otomatis mengekstrak string jawaban dari object LLM
)

# ==========================================
# 5. FUNGSI UTAMA (API UNTUK BACKEND)
# ==========================================
def ask_bot(question: str) -> str:
    """
    Fungsi ini dipanggil oleh endpoint Flask/Django.
    Menerima input string dari user, mereturn jawaban string dari AI.
    """
    try:
        # Karena kita pakai StrOutputParser, outputnya langsung berupa teks murni,
        # tidak perlu lagi manggil response["answer"]
        return rag_chain.invoke(question)
    except Exception as e:
        return f"Maaf, terjadi kesalahan pada sistem AI: {e}"

# ==========================================
# TES INTERNAL
# ==========================================
if __name__ == "__main__":
    print("\n" + "="*50)
    print("🤖 Bot FMIPA (LCEL Modern Architecture) Siap!")
    print("Ketik 'exit' untuk keluar.")
    print("="*50)

    while True:
        user_input = input("\nKamu: ")
        if user_input.lower() == 'exit':
            print("Mematikan bot... Sampai jumpa!")
            break

        jawaban = ask_bot(user_input)
        print(f"\nBot FMIPA: {jawaban}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Memuat Vector Database...
Inisialisasi Gemini API...

🤖 Bot FMIPA (LCEL Modern Architecture) Siap!
Ketik 'exit' untuk keluar.

Kamu: siapa rektor pertama UHO?

Bot FMIPA: Halo! Berdasarkan informasi yang saya miliki, UHO pertama kali dipimpin oleh Drs. Eddy Agussalim Mokodompit, M.A.

Kamu: kamu tau rektor pertamanya UHO?

Bot FMIPA: Tentu, saya tahu! UHO pertama kali dipimpin oleh Drs. Eddy Agussalim Mokodompit, M.A.

Kamu: siapa plt rektor sekarang

Bot FMIPA: Mohon maaf, saya belum memiliki informasi mengenai siapa Plt. Rektor saat ini berdasarkan konteks yang diberikan.

Kamu: kapan pembayaran UKT?

Bot FMIPA: Halo! Berdasarkan informasi yang saya miliki, saya belum menemukan jadwal pasti mengenai kapan pembayaran UKT dilakukan.

Yang tersedia adalah informasi mengenai "Pengumuman Pembayaran Uang Kuliah Tunggal (UKT) Dan Biaya Pemeriksaan Kesehatan (PEMKES) Calon Mahasiswa Baru Jalur Seleksi Nasional Berdasarkan Prestasi (SNBP) Tahun 2026" yang akan diumumkan pada 23 Mei 2026. Namu

KeyboardInterrupt: Interrupted by user

In [23]:
if "GOOGLE_API_KEY" in os.environ:
    del os.environ["GOOGLE_API_KEY"]

# Evaluasi Model

In [36]:
import numpy as np
from sentence_transformers import SentenceTransformer, util

# 1. Dataset Ground Truth (Diambil dari FAQ UHO dan Kalender Akademik)
test_data = [
    {
        "pertanyaan": "Apa itu Universitas Halu Oleo?",
        "referensi_jawaban": "Universitas Halu Oleo (UHO) adalah perguruan tinggi negeri di Kendari, Sulawesi Tenggara yang berdiri berdasarkan Keputusan Presiden RI Nomor 37 Tahun 1981."
    },
    {
        "pertanyaan": "Kapan registrasi akun SNPMB siswa untuk SNBP?",
        "referensi_jawaban": "Registrasi akun SNPMB siswa berlangsung dari 13 Januari 2025 sampai 18 Februari 2025."
    },
    {
        "pertanyaan": "Apakah FMIPA UHO memiliki Program Studi Oseanografi?",
        "referensi_jawaban": "Ya, FMIPA UHO memiliki Program Studi Oseanografi."
    },
    {
        "pertanyaan": "Berapa nomor telepon Universitas Halu Oleo?",
        "referensi_jawaban": "Nomor telepon UHO adalah (0401) 3190105."
    },
    {
        "pertanyaan": "Kapan pelaksanaan UTBK-SNBT?",
        "referensi_jawaban": "Pelaksanaan UTBK-SNBT berlangsung dari 23 April 2025 sampai 3 Mei 2025."
    }
]

In [37]:
# 2. Inisialisasi Evaluator Model (Bisa pakai model yang sama dengan pembuat Vector DB)
print("Memuat Evaluator Model...")
evaluator_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

def run_evaluation(test_dataset):
    print(f"\nMemulai evaluasi untuk {len(test_dataset)} pertanyaan...\n")
    print("-" * 50)

    total_score = 0
    results = []

    for idx, data in enumerate(test_dataset):
        q = data["pertanyaan"]
        ref_ans = data["referensi_jawaban"]

        # A. Dapatkan jawaban dari Chatbot RAG kita
        print(f"Mengeksekusi Q{idx+1}...")
        bot_ans = ask_bot(q)

        # B. Ubah teks jawaban ke dalam bentuk Vektor
        ref_embedding = evaluator_model.encode(ref_ans, convert_to_tensor=True)
        bot_embedding = evaluator_model.encode(bot_ans, convert_to_tensor=True)

        # C. Hitung Cosine Similarity (Skor Kemiripan Makna: 0.0 sampai 1.0)
        cosine_scores = util.cos_sim(ref_embedding, bot_embedding)
        score = cosine_scores.item()

        total_score += score

        # D. Simpan hasil evaluasi
        results.append({
            "Pertanyaan": q,
            "Referensi": ref_ans,
            "Jawaban Bot": bot_ans,
            "Skor Kemiripan": round(score, 3)
        })

        print(f"Q: {q}")
        print(f"Skor: {round(score, 3)}\n")

    # Hitung Rata-rata Skor Keseluruhan
    avg_score = total_score / len(test_dataset)
    print("=" * 50)
    print(f"EVALUASI SELESAI. Rata-rata Skor Kemiripan Semantik: {round(avg_score, 3)} / 1.0")
    print("=" * 50)

    return results, avg_score

if __name__ == "__main__":
    hasil_evaluasi, rata_rata = run_evaluation(test_data)

    # Kamu bisa menggunakan library Pandas untuk export 'hasil_evaluasi' ke Excel/CSV
    # agar rapi saat dimasukkan ke bab hasil pembahasan skripsi.

Memuat Evaluator Model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Memulai evaluasi untuk 5 pertanyaan...

--------------------------------------------------
Mengeksekusi Q1...
Q: Apa itu Universitas Halu Oleo?
Skor: 0.898

Mengeksekusi Q2...
Q: Kapan registrasi akun SNPMB siswa untuk SNBP?
Skor: 0.888

Mengeksekusi Q3...
Q: Apakah FMIPA UHO memiliki Program Studi Oseanografi?
Skor: 1.0

Mengeksekusi Q4...
Q: Berapa nomor telepon Universitas Halu Oleo?
Skor: 0.729

Mengeksekusi Q5...
Q: Kapan pelaksanaan UTBK-SNBT?
Skor: 0.928

EVALUASI SELESAI. Rata-rata Skor Kemiripan Semantik: 0.889 / 1.0
